In [ ]:
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
import requests

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool=DuckDuckGoSearchRun()

In [ ]:
@tool
def get_weather_data(city : str) ->str:
    weatherstack_api_key=""
    """
    This function fetches the current weather data for a given city
    """
    url=f'http://api.weatherstack.com/current?access_key={weatherstack_api_key}&query={city}'
    response=requests.get(url)
    return response.json()

In [ ]:
import os
os.environ['OPENAI_API_KEY']=''
os.environ['OPENAI_API_VERSION']=''
os.environ['OPENAI_AZURE_ENDPOINT']=''
os.environ['OPENAI_AZURE_MODEL']=''

In [ ]:
llm=AzureChatOpenAI(
        api_key=os.environ['OPENAI_API_KEY'],
        api_version=os.environ['OPENAI_API_VERSION'],
        azure_endpoint=os.environ['OPENAI_AZURE_ENDPOINT'],
        model_name=os.environ['OPENAI_AZURE_MODEL'],
        temperature=0.4,
        max_tokens=1000,
        seed=42,
)

In [ ]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

In [ ]:
from langchain.prompts import PromptTemplate

template = """
You are a reasoning agent that can use tools.

You must always follow this format:

Question: the input question you need to answer
Thought: your reasoning about what to do next
Action: the action to take, must be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (repeat Thought/Action/Action Input/Observation as needed)
Thought: I now know the final answer
Final Answer: the final answer to the original question

You have access to the following tools:
{tools}

Begin!

Question: {input} 
{agent_scratchpad}
"""

prompt = PromptTemplate(
    input_variables=["input", "tools", "tool_names", "agent_scratchpad"],
    template=template,
)

In [ ]:
agent=create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather_data],
    prompt=prompt
)

In [ ]:
agent_executor=AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True
)

In [ ]:
response=agent_executor.invoke({"input":"Find the capital of Madhya Pradesh, then find it's current weather  condition"})
print(response)
print(response['output'])